# ETL Pipeline: Data Cleaning and Preprocessing

This notebook implements the Extract, Transform, and Load (ETL) pipeline for the Mutual Fund Analytics Platform. It performs dataset inspection, validation, preprocessing, date parsing, and saves cleaned datasets for downstream analysis.

In [1]:
import pandas as pd
from pandas.api.types import is_datetime64_any_dtype, is_numeric_dtype

## Helper Functions

The following helper functions are reused throughout the notebook for dataset inspection, date conversion, and saving cleaned datasets.

In [21]:
def inspect_dataset(df):
    print("\n========== DATASET INFO ==========")
    df.info()
    print()
    print("Dataset Shape:", df.shape)
    print("\nNull values:\n", df.isnull().sum())
    print("\nDuplicate rows:", df.duplicated().sum())

    print("\nUnique values in categorical columns:")

    for col in df.select_dtypes(include=["object", "string"]).columns:

        unique_values = df[col].unique()

        print(f"\n{col} ({len(unique_values)} unique values):")

        if len(unique_values) <= 20:
            print(unique_values)
        else:
            print(unique_values[:20])
            print("...")


def parse_date(df, column):
    df[column] = pd.to_datetime(df[column])

    if is_datetime64_any_dtype(df[column]):
        print(f"The {column} column successfully parsed to datetime!")
    else:
        print(f"Failed to parse {column} to datetime!")

    return df


def save_processed(df, filename):
    df.to_csv(f"../data/processed/{filename}", index=False)
    print("Cleaned dataset saved successfully!")

## Cleaning NAV History Dataset

This section validates the historical NAV dataset before loading it into the processed directory.

In [19]:
def clean_nav_history():
    nav_history = pd.read_csv("../data/raw/02_nav_history.csv")
    inspect_dataset(nav_history)

    nav_history = parse_date(nav_history, "date")

    nav_history = nav_history.sort_values(["amfi_code", "date"])
    print("Data sorted according to AMFI codes and Date successfully!")

    invalid_nav = nav_history[nav_history["nav"] <= 0]

    if len(invalid_nav) == 0:
        print("All NAV values are greater than 0")
    else:
        print("Invalid NAV records found:\n", invalid_nav)

    save_processed(nav_history, "nav_history_cleaned.csv")

## Cleaning Investor Transactions

This section validates investor transaction records, checks transaction amounts, parses dates, and exports the cleaned dataset.

In [4]:
def clean_investor_transactions():
    inv_transactions = pd.read_csv("../data/raw/08_investor_transactions.csv")
    inspect_dataset(inv_transactions)

    inv_transactions = parse_date(inv_transactions,"transaction_date")

    print("Unique KYC status values:\n",inv_transactions["kyc_status"].unique())

    amount = inv_transactions[inv_transactions["amount_inr"] <= 0]

    if len(amount) == 0:
        print("All transaction amounts are greater than 0")
    else:
        print("Invalid amounts found:\n", amount)

    save_processed(inv_transactions,"transactions_cleaned.csv")

## Cleaning Scheme Performance Dataset

This section validates return values and expense ratios before exporting the cleaned dataset.

In [5]:
def clean_scheme_performance():
    scheme_performance = pd.read_csv("../data/raw/07_scheme_performance.csv")
    inspect_dataset(scheme_performance)

    return_yr = scheme_performance[["return_1yr_pct","return_3yr_pct","return_5yr_pct"]]

    for col in return_yr:

        if is_numeric_dtype(scheme_performance[col]):
            print(f"{col} is numeric.")
        else:
            print(f"{col} is NOT numeric.")

            print(f"\nInvalid values found in {col}:")

            for value in scheme_performance[col]:

                if not isinstance(value, (int, float)):
                    print(value)

    invalid_count = 0
    for index, row in scheme_performance.iterrows():

        exp = row["expense_ratio_pct"]

        if exp < 0.1 or exp > 2.5:
            invalid_count += 1
            print(f"{row['scheme_name']} -> Invalid Expense Ratio: {exp}")
    if invalid_count == 0:
        print("All expense ratios are within the valid range (0.1% to 2.5%).")
    
    save_processed(scheme_performance, "scheme_performance_cleaned.csv")

## Cleaning Fund Master Dataset

Load and inspect the master dataset containing mutual fund scheme information. Convert launch dates into a standardized datetime format and export the cleaned dataset.

In [10]:
def clean_fund_master():
    fund_master = pd.read_csv("../data/raw/01_fund_master.csv")
    inspect_dataset(fund_master)
    fund_master = parse_date(fund_master, "launch_date")
    save_processed(fund_master, "fund_master_cleaned.csv")

## Cleaning AUM by Fund House Dataset

Process Assets Under Management (AUM) data by validating date formats and preparing the dataset for trend analysis and dashboard reporting.

In [11]:
def clean_aum_by_fund_house():
    aum_by_fund_house = pd.read_csv("../data/raw/03_aum_by_fund_house.csv")
    inspect_dataset(aum_by_fund_house)
    aum_by_fund_house = parse_date(aum_by_fund_house,"date")
    save_processed(aum_by_fund_house, "aum_by_fund_house_cleaned.csv")

## Cleaning Monthly SIP Inflows Dataset

Inspect monthly SIP industry statistics, standardize date fields, and prepare the dataset for industry trend analysis and visualization.

In [12]:
def clean_monthly_sip_inflows():
    monthly_sip_inflows = pd.read_csv("../data/raw/04_monthly_sip_inflows.csv")
    inspect_dataset(monthly_sip_inflows)
    monthly_sip_inflows = parse_date(monthly_sip_inflows,"month")
    save_processed(monthly_sip_inflows, "monthly_sip_inflows_cleaned.csv")

## Cleaning Category Inflows Dataset

Validate category-wise mutual fund inflows and standardize date values to support comparative analysis across fund categories.

In [13]:
def clean_category_inflows():
    category_inflows = pd.read_csv("../data/raw/05_category_inflows.csv")
    inspect_dataset(category_inflows)
    category_inflows = parse_date(category_inflows,"month")
    save_processed(category_inflows, "category_inflows_cleaned.csv")

## Cleaning Industry Folio Count Dataset

Prepare industry folio statistics by validating the dataset structure and converting reporting periods into a consistent datetime format.

In [14]:
def clean_industry_folio_count():
    industry_folio_count = pd.read_csv("../data/raw/06_industry_folio_count.csv")
    inspect_dataset(industry_folio_count)
    industry_folio_count = parse_date(industry_folio_count,"month")
    save_processed(industry_folio_count, "industry_folio_count_cleaned.csv")

## Cleaning Portfolio Holdings Dataset

Inspect sector-wise portfolio holdings, validate portfolio dates, and generate a cleaned dataset for portfolio allocation analysis.

In [15]:
def clean_portfolio_holdings():
    portfolio_holdings = pd.read_csv("../data/raw/09_portfolio_holdings.csv")
    inspect_dataset(portfolio_holdings)
    portfolio_holdings = parse_date(portfolio_holdings,"portfolio_date")
    save_processed(portfolio_holdings, "portfolio_holdings_cleaned.csv")

## Cleaning Benchmark Indices Dataset

Process benchmark index data by validating dates and preparing the dataset for benchmark comparison and performance evaluation.

In [16]:
def clean_benchmark_indices():
    benchmark_indices = pd.read_csv("../data/raw/10_benchmark_indices.csv")
    inspect_dataset(benchmark_indices)
    benchmark_indices = parse_date(benchmark_indices,"date")
    save_processed(benchmark_indices, "benchmark_indices_cleaned.csv")

## Execute ETL Pipeline

Execute the required ETL functions to clean and preprocess the selected datasets. Individual cleaning functions can be enabled or disabled depending on the dataset being processed.

In [22]:
if __name__ == "__main__":
    clean_nav_history()
    # clean_investor_transactions()
    # clean_scheme_performance()
    # clean_fund_master()
    # clean_aum_by_fund_house()
    # clean_monthly_sip_inflows()
    # clean_category_inflows()
    # clean_industry_folio_count()
    # clean_portfolio_holdings()
    # clean_benchmark_indices()


========== DATASET INFO ==========
<class 'pandas.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   amfi_code  46000 non-null  int64  
 1   date       46000 non-null  str    
 2   nav        46000 non-null  float64
dtypes: float64(1), int64(1), str(1)
memory usage: 1.1 MB

Dataset Shape: (46000, 3)

Null values:
 amfi_code    0
date         0
nav          0
dtype: int64

Duplicate rows: 0

Unique values in categorical columns:

date (1150 unique values):
<StringArray>
['2022-01-03', '2022-01-04', '2022-01-05', '2022-01-06', '2022-01-07',
 '2022-01-10', '2022-01-11', '2022-01-12', '2022-01-13', '2022-01-14',
 '2022-01-17', '2022-01-18', '2022-01-19', '2022-01-20', '2022-01-21',
 '2022-01-24', '2022-01-25', '2022-01-26', '2022-01-27', '2022-01-28']
Length: 20, dtype: str
...
The date column successfully parsed to datetime!
Data sorted according to AMFI codes and Date succ

## Summary

The ETL pipeline successfully inspected, validated, and standardized the raw mutual fund datasets. All cleaned datasets were exported to the data/processed directory and served as the primary data source for the SQLite database, analytical notebooks, and Power BI dashboard. The modular design of the pipeline also makes it reusable for future data updates.